# Phase 2–4 — Train Semantic Energy & Entropy Probes

This is the **main training notebook**. It is self-contained and presentable.

**Prerequisites**: Run `01_generate_dataset.ipynb` first to produce the dataset pkl.

**What this notebook trains**:
| Probe | Input | Labels | Deployment mode |
|---|---|---|---|
| SLT Energy probe | Hidden state at 2nd-to-last generated token | Binarized `energy_score_raw` | Post-generation |
| TBG Energy probe | Hidden state at last prompt token | Binarized `energy_score_raw` | Pre-generation |
| SLT Entropy probe | Hidden state at 2nd-to-last generated token | Binarized `entropy_score_raw` | Post-generation |
| TBG Entropy probe | Hidden state at last prompt token | Binarized `entropy_score_raw` | Pre-generation |

**Hallucination risk inference direction**:
- Energy probes: `1 - probe_score` (energy = confidence, invert for risk)
- Entropy probes: `probe_score` directly (entropy = uncertainty = risk)

## Section 1 — Naming Conventions

See `00_preflight.ipynb` Section 1 for the full naming table. Quick reference:

| Variable | Meaning |
|---|---|
| `t` prefix | TBG token (last prompt token, pre-generation) |
| `s` prefix | SLT token (second-to-last generated token, post-generation) |
| `_a` suffix | accuracy probe (correctness labels) |
| `_b` suffix | binarized teacher probe (teacher distribution labels) |
| `energy_label` | 1 = high energy (confident), 0 = low energy |
| `entropy_label` | 1 = high entropy (uncertain), 0 = low entropy |

## Section 2 — Imports

In [1]:
import os
import sys
import pickle
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR    = os.path.join(REPO_ROOT, 'backend', 'data')
MODELS_DIR  = os.path.join(REPO_ROOT, 'backend', 'models')
FIGS_DIR    = os.path.join(REPO_ROOT, 'notebooks', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

np.random.seed(42)
print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")

Repo root : d:\Github Repositories\SemanticEnergy
Data dir  : d:\Github Repositories\SemanticEnergy\backend\data
Models dir: d:\Github Repositories\SemanticEnergy\backend\models


In [2]:
import sklearn
import scipy
print(f"sklearn {sklearn.__version__}, scipy {scipy.__version__}, numpy {np.__version__}")

sklearn 1.8.0, scipy 1.17.1, numpy 2.3.5


## Section 3 — Dataset Class and Splits

In [3]:
class ProbeDataset:
    """
    Container for probe training data. Stores all record fields as arrays
    and provides easy access to hidden states and labels.
    Modeled on SEP's Dataset class pattern.
    """
    def __init__(self, records, name='train'):
        self.name    = name
        self.N       = len(records)

        # Raw scores (continuous teacher outputs)
        self.energy_score_raw  = np.array([r['energy_score_raw']  for r in records])   # (N,) in [0,1]
        self.entropy_score_raw = np.array([r['entropy_score_raw'] for r in records])   # (N,) >= 0

        # Correctness
        self.correctness = np.array([r['correctness'] for r in records])  # (N,) in {0.0, 1.0}

        # Hidden states: (N, num_layers, hidden_dim)
        self.tbg_states = np.stack([r['emb_last_tok_before_gen'] for r in records])
        self.slt_states = np.stack([r['emb_tok_before_eos']      for r in records])

        self.num_layers = self.tbg_states.shape[1]   # e.g. 33 for Llama 3.1 8B
        self.hidden_dim = self.tbg_states.shape[2]   # e.g. 4096

        # Logit features: (N, 4) — primary logit-centered features
        # NOTE: mean/min/std_logit_margin are excluded because output_scores=True in transformers 5.x
        # returns post-filtered logits where top2 is -inf, making all margins inf.
        # Fix applied to engine.py (output_logits=True) for future data collection.
        # Current dataset uses only the 4 valid features below.
        self.logit_feats = np.array([
            [
                r['logit_feats']['mean_chosen_logit'],
                r['logit_feats']['min_chosen_logit'],
                r['logit_feats']['std_chosen_logit'],
                float(r['logit_feats']['answer_len']),
            ]
            for r in records
        ], dtype=np.float64)   # (N, 4)

        # Replace any residual inf/nan just in case
        self.logit_feats = np.nan_to_num(self.logit_feats, nan=0.0, posinf=0.0, neginf=0.0)

        # Binarized labels (set by Section 4)
        self.energy_label  = None   # (N,) — 1 = high energy (confident)
        self.entropy_label = None   # (N,) — 1 = high entropy (uncertain)

        print(f"ProbeDataset '{name}': N={self.N}, layers={self.num_layers}, hidden_dim={self.hidden_dim}")
        print(f"  TBG shape:         {self.tbg_states.shape}")
        print(f"  SLT shape:         {self.slt_states.shape}")
        print(f"  Logit feats shape: {self.logit_feats.shape}  (4 features: mean/min/std logit + answer_len)")

print("ProbeDataset class defined.")

ProbeDataset class defined.


In [4]:
# Load dataset
DATASET_PATH = os.path.join(DATA_DIR, 'probe_dataset_llama3-8b_triviaqa.pkl')
print(f"Loading: {DATASET_PATH}")
with open(DATASET_PATH, 'rb') as f:
    all_records = pickle.load(f)

print(f"Records loaded: {len(all_records)}")

# Split: 80% train, 10% val, 10% test
N = len(all_records)
idx = np.random.permutation(N)
n_train = int(0.80 * N)
n_val   = int(0.10 * N)
n_test  = N - n_train - n_val

train_records = [all_records[i] for i in idx[:n_train]]
val_records   = [all_records[i] for i in idx[n_train:n_train+n_val]]
test_records  = [all_records[i] for i in idx[n_train+n_val:]]

print(f"Split: train={n_train}, val={n_val}, test={n_test}")

D_train = ProbeDataset(train_records, name='train')
D_val   = ProbeDataset(val_records,   name='val')
D_test  = ProbeDataset(test_records,  name='test')

Loading: d:\Github Repositories\SemanticEnergy\backend\data\probe_dataset_llama3-8b_triviaqa.pkl
Records loaded: 500
Split: train=400, val=50, test=50
ProbeDataset 'train': N=400, layers=33, hidden_dim=4096
  TBG shape:         (400, 33, 4096)
  SLT shape:         (400, 33, 4096)
  Logit feats shape: (400, 4)  (4 features: mean/min/std logit + answer_len)
ProbeDataset 'val': N=50, layers=33, hidden_dim=4096
  TBG shape:         (50, 33, 4096)
  SLT shape:         (50, 33, 4096)
  Logit feats shape: (50, 4)  (4 features: mean/min/std logit + answer_len)
ProbeDataset 'test': N=50, layers=33, hidden_dim=4096
  TBG shape:         (50, 33, 4096)
  SLT shape:         (50, 33, 4096)
  Logit feats shape: (50, 4)  (4 features: mean/min/std logit + answer_len)


In [5]:
# ── PATCH: sanitize logit_feats for all splits ─────────────────────────────
# The dataset was collected before engine.py output_logits=True fix, so
# mean/std/min_logit_margin columns contain inf/nan (top2 from post-filtered
# scores is -inf, making margin = inf).
# This cell strips those columns and cleans any residual inf/nan.
# Safe to run regardless of which ProbeDataset version loaded the data.
import numpy as np

VALID_LOGIT_COLS = 4   # mean_chosen_logit, min_chosen_logit, std_chosen_logit, answer_len

for D in [D_train, D_val, D_test]:
    # Keep only the first 4 columns if more were loaded
    if D.logit_feats.shape[1] > VALID_LOGIT_COLS:
        D.logit_feats = D.logit_feats[:, :VALID_LOGIT_COLS]
    # Replace any residual inf/nan
    D.logit_feats = np.nan_to_num(D.logit_feats, nan=0.0, posinf=0.0, neginf=0.0)

print("logit_feats patched for all splits.")
for D in [D_train, D_val, D_test]:
    n_inf = np.sum(np.isinf(D.logit_feats))
    n_nan = np.sum(np.isnan(D.logit_feats))
    print(f"  {D.name:6s}: shape={D.logit_feats.shape}  inf={n_inf}  nan={n_nan}")

logit_feats patched for all splits.
  train : shape=(400, 4)  inf=0  nan=0
  val   : shape=(50, 4)  inf=0  nan=0
  test  : shape=(50, 4)  inf=0  nan=0


## Section 4 — Label Binarization

We use **teacher-distribution-only thresholding** (no correctness labels involved).
Threshold is chosen to minimize the weighted within-group MSE — exactly as in SEP.

```
MSE(t) = Σ_{s < t} (s - mean_low)² + Σ_{s ≥ t} (s - mean_high)²
```

The threshold that minimizes this assigns the most internally consistent binary labels.

In [6]:
def find_best_threshold(scores, label='scores'):
    """
    SEP-style: find threshold minimizing sum of squared errors within each group.
    Sweeps percentile-based thresholds from 10th to 90th percentile.
    Returns the best threshold and plots the MSE curve.
    """
    best_thresh = None
    best_mse    = float('inf')
    thresholds  = []
    mses        = []
    
    for pct in np.arange(10, 91, 1):
        thresh = np.percentile(scores, pct)
        g0 = scores[scores <  thresh]
        g1 = scores[scores >= thresh]
        if len(g0) == 0 or len(g1) == 0:
            continue
        mse = (len(g0) * g0.var() + len(g1) * g1.var()) / len(scores)
        thresholds.append(thresh)
        mses.append(mse)
        if mse < best_mse:
            best_mse    = mse
            best_thresh = thresh
    
    return best_thresh, np.array(thresholds), np.array(mses)


def binarize(scores, threshold):
    """Return binary labels: 1 if score >= threshold, else 0."""
    return (scores >= threshold).astype(int)


print("Binarization functions defined.")

Binarization functions defined.


In [7]:
# Compute thresholds on TRAINING split only
# (Apply same thresholds to val and test)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Energy threshold
T_energy, energy_thresholds, energy_mses = find_best_threshold(
    D_train.energy_score_raw, label='energy'
)
axes[0].plot(energy_thresholds, energy_mses, color='steelblue', linewidth=2)
axes[0].axvline(T_energy, color='red', linestyle='--', label=f'threshold = {T_energy:.4f}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Within-group MSE')
axes[0].set_title('Energy Score — Binarization Threshold Selection')
axes[0].legend()

# Entropy threshold
T_entropy, entropy_thresholds, entropy_mses = find_best_threshold(
    D_train.entropy_score_raw, label='entropy'
)
axes[1].plot(entropy_thresholds, entropy_mses, color='darkorange', linewidth=2)
axes[1].axvline(T_entropy, color='red', linestyle='--', label=f'threshold = {T_entropy:.4f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Within-group MSE')
axes[1].set_title('Entropy Score — Binarization Threshold Selection')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, 'binarization_threshold_sweep.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f"Energy threshold  T_energy  = {T_energy:.4f}")
print(f"Entropy threshold T_entropy = {T_entropy:.4f}")

Energy threshold  T_energy  = 0.7504
Entropy threshold T_entropy = 0.2052


In [8]:
# Apply binarization to all splits
for D in [D_train, D_val, D_test]:
    D.energy_label  = binarize(D.energy_score_raw,  T_energy)
    D.entropy_label = binarize(D.entropy_score_raw, T_entropy)

print("Energy label balance:")
for D in [D_train, D_val, D_test]:
    pos = D.energy_label.mean()
    print(f"  {D.name:6s}: {pos:.3f} positive (high energy)")

print("\nEntropy label balance:")
for D in [D_train, D_val, D_test]:
    pos = D.entropy_label.mean()
    print(f"  {D.name:6s}: {pos:.3f} positive (high entropy)")

Energy label balance:
  train : 0.740 positive (high energy)
  val   : 0.760 positive (high energy)
  test  : 0.820 positive (high energy)

Entropy label balance:
  train : 0.410 positive (high entropy)
  val   : 0.380 positive (high entropy)
  test  : 0.280 positive (high entropy)


## Section 5 — Helper Functions

In [9]:
def get_layer_X(D, layer_idx, token='slt'):
    """Extract hidden state at a single layer. Returns (N, hidden_dim)."""
    states = D.slt_states if token == 'slt' else D.tbg_states
    return states[:, layer_idx, :]   # (N, hidden_dim)


def get_range_X(D, layer_range, token='slt'):
    """Concatenate hidden states across a layer range. Returns (N, len(range) * hidden_dim)."""
    l_start, l_end = layer_range
    states = D.slt_states if token == 'slt' else D.tbg_states
    return states[:, l_start:l_end, :].reshape(D.N, -1)   # (N, range_len * hidden_dim)


def clean_X(X):
    """Replace any inf/nan with 0. Defensive guard before passing to sklearn."""
    X = np.array(X, dtype=np.float64)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


def sklearn_train_eval(X_train, y_train, X_val, y_val, scale=True):
    """Train a LogisticRegression probe and return val AUROC."""
    X_train = clean_X(X_train)
    X_val   = clean_X(X_val)
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)
    else:
        scaler = None
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(X_train, y_train)
    y_score = probe.predict_proba(X_val)[:, 1]
    auroc = roc_auc_score(y_val, y_score)
    return probe, scaler, auroc


def bootstrap_auroc(probe, scaler, X_test, y_test, n_boot=1000, ci=0.95):
    """Bootstrap 95% CI for AUROC on test set."""
    if scaler is not None:
        X_test = scaler.transform(clean_X(X_test))
    y_score = probe.predict_proba(X_test)[:, 1]
    base_auroc = roc_auc_score(y_test, y_score)

    boot_aurocs = []
    rng = np.random.default_rng(0)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_test), len(y_test))
        if len(np.unique(y_test[idx])) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_test[idx], y_score[idx]))

    alpha = (1 - ci) / 2
    lo = np.percentile(boot_aurocs, alpha * 100)
    hi = np.percentile(boot_aurocs, (1 - alpha) * 100)
    return {'mean': base_auroc, 'lo': lo, 'hi': hi}


def save_fig(name):
    path = os.path.join(FIGS_DIR, f'{name}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    print(f"Saved: {path}")


print("Helper functions defined.")

Helper functions defined.


## Section 6 — Per-Layer AUROC Sweep — Energy Probe

Full mandatory sweep across **all** layers. Do not narrow the layer range before seeing these plots.

In [10]:
num_layers = D_train.num_layers
print(f"Total layers: {num_layers}")
print(f"Running per-layer AUROC sweep for Energy probe (SLT and TBG)...")
print(f"This will train {num_layers * 2} probes. Expected time: several minutes.")

Total layers: 33
Running per-layer AUROC sweep for Energy probe (SLT and TBG)...
This will train 66 probes. Expected time: several minutes.


In [11]:
energy_slt_aurocs = []
energy_tbg_aurocs = []

for layer in tqdm(range(num_layers), desc='Energy probe — layer sweep'):
    X_train_slt = get_layer_X(D_train, layer, 'slt')
    X_val_slt   = get_layer_X(D_val,   layer, 'slt')
    _, _, auroc_slt = sklearn_train_eval(X_train_slt, D_train.energy_label, X_val_slt, D_val.energy_label)
    energy_slt_aurocs.append(auroc_slt)
    
    X_train_tbg = get_layer_X(D_train, layer, 'tbg')
    X_val_tbg   = get_layer_X(D_val,   layer, 'tbg')
    _, _, auroc_tbg = sklearn_train_eval(X_train_tbg, D_train.energy_label, X_val_tbg, D_val.energy_label)
    energy_tbg_aurocs.append(auroc_tbg)

print(f"SLT Energy: best layer = {np.argmax(energy_slt_aurocs)}, AUROC = {max(energy_slt_aurocs):.4f}")
print(f"TBG Energy: best layer = {np.argmax(energy_tbg_aurocs)}, AUROC = {max(energy_tbg_aurocs):.4f}")

Energy probe — layer sweep:   0%|          | 0/33 [00:00<?, ?it/s]

SLT Energy: best layer = 19, AUROC = 0.7412
TBG Energy: best layer = 14, AUROC = 0.8706


In [12]:
fig, ax = plt.subplots(figsize=(12, 5))
layers = list(range(num_layers))
ax.plot(layers, energy_slt_aurocs, marker='o', markersize=3, label='SLT (post-generation)', color='steelblue')
ax.plot(layers, energy_tbg_aurocs, marker='s', markersize=3, label='TBG (pre-generation)',  color='seagreen',  linestyle='--')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance (0.5)')
ax.set_xlabel('Layer index')
ax.set_ylabel('Validation AUROC')
ax.set_title('Energy Probe — Per-Layer AUROC (Llama 3.1 8B Instruct)')
ax.legend()
ax.set_ylim(0.4, 1.0)
ax.set_xticks(range(0, num_layers, 2))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig('energy_probe_layer_sweep')
plt.show()

Saved: d:\Github Repositories\SemanticEnergy\notebooks\figures\energy_probe_layer_sweep.png


## Section 7 — Per-Layer AUROC Sweep — Entropy Probe

In [13]:
entropy_slt_aurocs = []
entropy_tbg_aurocs = []

for layer in tqdm(range(num_layers), desc='Entropy probe — layer sweep'):
    X_train_slt = get_layer_X(D_train, layer, 'slt')
    X_val_slt   = get_layer_X(D_val,   layer, 'slt')
    _, _, auroc_slt = sklearn_train_eval(X_train_slt, D_train.entropy_label, X_val_slt, D_val.entropy_label)
    entropy_slt_aurocs.append(auroc_slt)
    
    X_train_tbg = get_layer_X(D_train, layer, 'tbg')
    X_val_tbg   = get_layer_X(D_val,   layer, 'tbg')
    _, _, auroc_tbg = sklearn_train_eval(X_train_tbg, D_train.entropy_label, X_val_tbg, D_val.entropy_label)
    entropy_tbg_aurocs.append(auroc_tbg)

print(f"SLT Entropy: best layer = {np.argmax(entropy_slt_aurocs)}, AUROC = {max(entropy_slt_aurocs):.4f}")
print(f"TBG Entropy: best layer = {np.argmax(entropy_tbg_aurocs)}, AUROC = {max(entropy_tbg_aurocs):.4f}")

Entropy probe — layer sweep:   0%|          | 0/33 [00:00<?, ?it/s]

SLT Entropy: best layer = 20, AUROC = 0.7929
TBG Entropy: best layer = 24, AUROC = 0.8693


In [14]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(layers, entropy_slt_aurocs, marker='o', markersize=3, label='SLT (post-generation)', color='darkorange')
ax.plot(layers, entropy_tbg_aurocs, marker='s', markersize=3, label='TBG (pre-generation)',  color='tomato',    linestyle='--')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance (0.5)')
ax.set_xlabel('Layer index')
ax.set_ylabel('Validation AUROC')
ax.set_title('Entropy Probe — Per-Layer AUROC (Llama 3.1 8B Instruct)')
ax.legend()
ax.set_ylim(0.4, 1.0)
ax.set_xticks(range(0, num_layers, 2))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig('entropy_probe_layer_sweep')
plt.show()

Saved: d:\Github Repositories\SemanticEnergy\notebooks\figures\entropy_probe_layer_sweep.png


## Section 8 — Layer Range Selection

In [15]:
def decide_layer_range(auroc_list, window_sizes=[4, 8, 16], min_window=4):
    """
    Find the contiguous layer window with the highest mean AUROC.
    Returns (best_mean, (start, end)) where range is [start, end).
    """
    aucs = np.array(auroc_list)
    best_mean  = -np.inf
    best_range = (0, min_window)
    
    for window in window_sizes:
        for start in range(len(aucs) - window + 1):
            end = start + window
            mean_auc = aucs[start:end].mean()
            if mean_auc > best_mean:
                best_mean  = mean_auc
                best_range = (start, end)
    
    return best_mean, best_range


best_energy_slt_mean,  best_energy_slt_range  = decide_layer_range(energy_slt_aurocs)
best_energy_tbg_mean,  best_energy_tbg_range  = decide_layer_range(energy_tbg_aurocs)
best_entropy_slt_mean, best_entropy_slt_range = decide_layer_range(entropy_slt_aurocs)
best_entropy_tbg_mean, best_entropy_tbg_range = decide_layer_range(entropy_tbg_aurocs)

print("Best layer ranges (contiguous window with highest mean validation AUROC):")
print(f"  Energy  SLT: layers {best_energy_slt_range}   mean AUROC = {best_energy_slt_mean:.4f}")
print(f"  Energy  TBG: layers {best_energy_tbg_range}   mean AUROC = {best_energy_tbg_mean:.4f}")
print(f"  Entropy SLT: layers {best_entropy_slt_range}  mean AUROC = {best_entropy_slt_mean:.4f}")
print(f"  Entropy TBG: layers {best_entropy_tbg_range}  mean AUROC = {best_entropy_tbg_mean:.4f}")

Best layer ranges (contiguous window with highest mean validation AUROC):
  Energy  SLT: layers (17, 21)   mean AUROC = 0.7270
  Energy  TBG: layers (28, 32)   mean AUROC = 0.7840
  Entropy SLT: layers (20, 24)  mean AUROC = 0.7725
  Entropy TBG: layers (21, 25)  mean AUROC = 0.8523


## Section 9 — Feature Ablation

Train each probe three ways:
1. Hidden states only (best layer range)
2. Primary logit features only (7 scalars)
3. Hidden states + logit features (concatenated)

Report validation AUROC to determine if logit features add value over hidden states.

In [16]:
def ablation_eval(D_train, D_val, label_key, token, layer_range):
    """
    Run three ablations: hidden-only, logit-only, hidden+logit.
    Returns dict of AUROC values.
    """
    y_train = getattr(D_train, label_key)
    y_val   = getattr(D_val,   label_key)
    
    # 1. Hidden states only
    X_train_h = get_range_X(D_train, layer_range, token)
    X_val_h   = get_range_X(D_val,   layer_range, token)
    _, _, auc_h = sklearn_train_eval(X_train_h, y_train, X_val_h, y_val)
    
    # 2. Logit features only
    _, _, auc_l = sklearn_train_eval(D_train.logit_feats, y_train, D_val.logit_feats, y_val)
    
    # 3. Hidden + logit features
    scaler_h = StandardScaler()
    X_train_h_s = scaler_h.fit_transform(X_train_h)
    X_val_h_s   = scaler_h.transform(X_val_h)
    scaler_l = StandardScaler()
    X_train_l_s = scaler_l.fit_transform(D_train.logit_feats)
    X_val_l_s   = scaler_l.transform(D_val.logit_feats)
    X_train_hl = np.concatenate([X_train_h_s, X_train_l_s], axis=1)
    X_val_hl   = np.concatenate([X_val_h_s,   X_val_l_s],   axis=1)
    probe_hl = LogisticRegression(max_iter=1000, C=1.0)
    probe_hl.fit(X_train_hl, y_train)
    auc_hl = roc_auc_score(y_val, probe_hl.predict_proba(X_val_hl)[:, 1])
    
    return {'hidden_only': auc_h, 'logit_only': auc_l, 'hidden_logit': auc_hl}


print("Ablation table (Validation AUROC):")
print(f"{'Probe':<22} {'Hidden only':>13} {'Logit only':>11} {'Hidden+Logit':>13}")
print("-" * 62)

ablation_results = {}
for name, label_key, token, layer_range in [
    ('Energy SLT',  'energy_label',  'slt', best_energy_slt_range),
    ('Energy TBG',  'energy_label',  'tbg', best_energy_tbg_range),
    ('Entropy SLT', 'entropy_label', 'slt', best_entropy_slt_range),
    ('Entropy TBG', 'entropy_label', 'tbg', best_entropy_tbg_range),
]:
    res = ablation_eval(D_train, D_val, label_key, token, layer_range)
    ablation_results[name] = res
    print(f"{name:<22} {res['hidden_only']:>13.4f} {res['logit_only']:>11.4f} {res['hidden_logit']:>13.4f}")

Ablation table (Validation AUROC):
Probe                    Hidden only  Logit only  Hidden+Logit
--------------------------------------------------------------
Energy SLT                    0.7281      0.9232        0.7346
Energy TBG                    0.7851      0.9232        0.7851
Entropy SLT                   0.7657      0.9066        0.7742
Entropy TBG                   0.8676      0.9066        0.8761


## Section 10 — In-Distribution Evaluation

Train on train split, evaluate on test split. Report AUROC with 95% bootstrap CI.

In [17]:
# Train final probes on train, evaluate on test
id_results = {}
trained_probes = {}

configs = [
    ('slt_energy',  'energy_label',  'slt', best_energy_slt_range),
    ('tbg_energy',  'energy_label',  'tbg', best_energy_tbg_range),
    ('slt_entropy', 'entropy_label', 'slt', best_entropy_slt_range),
    ('tbg_entropy', 'entropy_label', 'tbg', best_entropy_tbg_range),
]

for probe_name, label_key, token, layer_range in configs:
    y_train = getattr(D_train, label_key)
    y_test  = getattr(D_test,  label_key)
    
    X_train = get_range_X(D_train, layer_range, token)
    X_test  = get_range_X(D_test,  layer_range, token)
    
    probe, scaler, val_auroc = sklearn_train_eval(
        X_train, y_train,
        get_range_X(D_val, layer_range, token), getattr(D_val, label_key)
    )
    
    ci = bootstrap_auroc(probe, scaler, X_test, y_test)
    id_results[probe_name] = ci
    trained_probes[probe_name] = {'probe': probe, 'scaler': scaler, 'layer_range': layer_range, 'token': token}
    print(f"{probe_name:<15}: AUROC = {ci['mean']:.4f}  95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]")

slt_energy     : AUROC = 0.6667  95% CI [0.4742, 0.8370]
tbg_energy     : AUROC = 0.7480  95% CI [0.5151, 0.9377]
slt_entropy    : AUROC = 0.7877  95% CI [0.6547, 0.8949]
tbg_entropy    : AUROC = 0.7857  95% CI [0.6389, 0.8988]


## Section 11 — Teacher Fidelity

Spearman ρ between the probe's continuous output and the raw teacher score.
High ρ means the probe is a good approximation of the teacher.

In [18]:
print("Teacher Fidelity — Spearman ρ between probe score and raw teacher score")
print(f"{'Probe':<15} {'Teacher':<20} {'Spearman ρ':>12} {'p-value':>12}")
print("-" * 62)

for probe_name, label_key, token, layer_range in configs:
    teacher_key = 'energy_score_raw' if 'energy' in probe_name else 'entropy_score_raw'
    raw_teacher = getattr(D_test, teacher_key)
    
    X_test  = get_range_X(D_test, layer_range, token)
    p_info  = trained_probes[probe_name]
    X_test_s = p_info['scaler'].transform(X_test)
    probe_score = p_info['probe'].predict_proba(X_test_s)[:, 1]
    
    rho, p = spearmanr(probe_score, raw_teacher)
    print(f"{probe_name:<15} {teacher_key:<20} {rho:>12.4f} {p:>12.2e}")

Teacher Fidelity — Spearman ρ between probe score and raw teacher score
Probe           Teacher                Spearman ρ      p-value
--------------------------------------------------------------
slt_energy      energy_score_raw           0.2507     7.91e-02
tbg_energy      energy_score_raw           0.2616     6.64e-02
slt_entropy     entropy_score_raw          0.4330     1.68e-03
tbg_entropy     entropy_score_raw          0.4329     1.69e-03


## Section 12 — Full Teacher Upper Bounds

The full teachers serve as **upper bounds** for the probe AUROC — they use the expensive
multi-sample clustering process that the probes are approximating.

In [19]:
# Upper bound: full energy teacher
# Hallucination AUROC: 1 - energy_score_raw predicts 1 - correctness
y_true_hallucination = 1 - D_test.correctness  # 1 = hallucination, 0 = correct

energy_upper_auroc  = roc_auc_score(y_true_hallucination, 1 - D_test.energy_score_raw)
entropy_upper_auroc = roc_auc_score(y_true_hallucination, D_test.entropy_score_raw)

print("Full teacher upper bounds (test set):")
print(f"  Energy teacher  AUROC: {energy_upper_auroc:.4f}  (uses 1 - energy_score_raw)")
print(f"  Entropy teacher AUROC: {entropy_upper_auroc:.4f}  (uses entropy_score_raw directly)")
print()
print("Note: probes approximate these with a single forward pass instead of 5-sample generation + clustering.")

Full teacher upper bounds (test set):
  Energy teacher  AUROC: 0.7103  (uses 1 - energy_score_raw)
  Entropy teacher AUROC: 0.7143  (uses entropy_score_raw directly)

Note: probes approximate these with a single forward pass instead of 5-sample generation + clustering.


## Section 13 — Performance Comparison Bar Plot

In [20]:
# Collect AUROC values for all systems for hallucination detection
# All scores oriented as: higher = higher hallucination risk

def get_probe_hallucination_auroc(probe_name, probe_info, D_test, y_true_hall):
    """Compute hallucination AUROC for a probe (handling orientation)."""
    X_test  = get_range_X(D_test, probe_info['layer_range'], probe_info['token'])
    X_test_s = probe_info['scaler'].transform(X_test)
    probe_score = probe_info['probe'].predict_proba(X_test_s)[:, 1]
    
    # Energy probe = confidence → invert for hallucination risk
    if 'energy' in probe_name:
        risk_score = 1 - probe_score
    else:
        risk_score = probe_score  # entropy probe = uncertainty = risk directly
    
    return roc_auc_score(y_true_hall, risk_score)


# Logit features only baseline
logit_probe_e = LogisticRegression(max_iter=1000)
logit_probe_e.fit(D_train.logit_feats, D_train.energy_label)
logit_score_e = logit_probe_e.predict_proba(D_test.logit_feats)[:, 1]
logit_auroc = roc_auc_score(y_true_hallucination, 1 - logit_score_e)

systems = [
    ('Energy teacher (upper bound)',  energy_upper_auroc),
    ('Entropy teacher (upper bound)', entropy_upper_auroc),
    ('SLT energy probe',              get_probe_hallucination_auroc('slt_energy',  trained_probes['slt_energy'],  D_test, y_true_hallucination)),
    ('TBG energy probe',              get_probe_hallucination_auroc('tbg_energy',  trained_probes['tbg_energy'],  D_test, y_true_hallucination)),
    ('SLT entropy probe',             get_probe_hallucination_auroc('slt_entropy', trained_probes['slt_entropy'], D_test, y_true_hallucination)),
    ('TBG entropy probe',             get_probe_hallucination_auroc('tbg_entropy', trained_probes['tbg_entropy'], D_test, y_true_hallucination)),
    ('Logit features only',           logit_auroc),
]

print("Hallucination Detection AUROC — All Systems (test set)")
print(f"{'System':<35} {'AUROC':>8}")
print("-" * 45)
for name, auroc in systems:
    print(f"{name:<35} {auroc:>8.4f}")

Hallucination Detection AUROC — All Systems (test set)
System                                 AUROC
---------------------------------------------
Energy teacher (upper bound)          0.7103
Entropy teacher (upper bound)         0.7143
SLT energy probe                      0.7163
TBG energy probe                      0.6409
SLT entropy probe                     0.6726
TBG entropy probe                     0.6806
Logit features only                   0.8075


In [21]:
# Bar plot — modeled on SEP's plot_performance_barplots()
fig, ax = plt.subplots(figsize=(12, 5))

names  = [s[0] for s in systems]
values = [s[1] for s in systems]
colors = ['#2ecc71', '#27ae60', '#3498db', '#2980b9', '#e67e22', '#d35400', '#95a5a6']

bars = ax.barh(names, values, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0.5, color='black', linestyle=':', linewidth=1)
ax.set_xlabel('Hallucination Detection AUROC')
ax.set_title('Semantic Energy Probes — Performance Comparison (TriviaQA, Llama 3.1 8B Instruct)')
ax.set_xlim(0.3, 1.05)

for bar, val in zip(bars, values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
save_fig('performance_comparison_barplot')
plt.show()

Saved: d:\Github Repositories\SemanticEnergy\notebooks\figures\performance_comparison_barplot.png


## Section 14 — Cross-Signal Analysis

Are Energy and Entropy probes measuring the same thing or complementary information?

In [22]:
# Spearman ρ between energy_score_raw and entropy_score_raw
rho_cross, p_cross = spearmanr(D_test.energy_score_raw, D_test.entropy_score_raw)
print(f"Spearman ρ(energy_raw, entropy_raw): {rho_cross:.4f}  (p={p_cross:.2e})")
print(f"Expected: negative (high energy = low entropy = same signal in opposite directions)")
print()

# Compare probe predictions vs same ground truth
slt_energy_risk  = get_probe_hallucination_auroc('slt_energy',  trained_probes['slt_energy'],  D_test, y_true_hallucination)
slt_entropy_risk = get_probe_hallucination_auroc('slt_entropy', trained_probes['slt_entropy'], D_test, y_true_hallucination)
print(f"SLT Energy probe AUROC (hallucination):  {slt_energy_risk:.4f}")
print(f"SLT Entropy probe AUROC (hallucination): {slt_entropy_risk:.4f}")
print()

# Are probe predictions correlated with each other?
X_slt_e = get_range_X(D_test, best_energy_slt_range, 'slt')
X_slt_h = get_range_X(D_test, best_entropy_slt_range, 'slt')
e_score = trained_probes['slt_energy']['probe'].predict_proba(
    trained_probes['slt_energy']['scaler'].transform(X_slt_e))[:, 1]
h_score = trained_probes['slt_entropy']['probe'].predict_proba(
    trained_probes['slt_entropy']['scaler'].transform(X_slt_h))[:, 1]
rho_probe_cross, _ = spearmanr(e_score, h_score)
print(f"Spearman ρ(energy_probe_score, entropy_probe_score): {rho_probe_cross:.4f}")

Spearman ρ(energy_raw, entropy_raw): -0.9958  (p=1.48e-51)
Expected: negative (high energy = low entropy = same signal in opposite directions)

SLT Energy probe AUROC (hallucination):  0.7163
SLT Entropy probe AUROC (hallucination): 0.6726

Spearman ρ(energy_probe_score, entropy_probe_score): -0.7444


## Section 15 — Efficiency Comparison

In [23]:
import time

# Time measurements (single example)
# These are rough wall-clock estimates — run on a single question

print("Efficiency comparison (wall-clock, single example):")
print()

# 1. Full energy teacher
test_q = 'What is the capital of France?'

try:
    import math
    from engine import SemanticEngine, cal_flow, sum_normalize
    print("[1/3] Full energy teacher (5 generations + clustering)...")
    t0 = time.time()
    gen_data = engine.generate_responses(test_q, num_samples=5)
    answers = [d['answer'] for d in gen_data]
    clusters = engine.find_semantic_clusters(test_q, answers)
    probs_se, logits_se = cal_flow([d['probs'] for d in gen_data], [d['logits'] for d in gen_data], clusters)
    _ = sum_normalize(logits_se)
    t_teacher = time.time() - t0
    print(f"   Full teacher: {t_teacher:.1f}s")
except:
    print("   (engine not available in this run — estimates only)")
    t_teacher = None

print()
print("Summary (estimates):")
print(f"  Full energy teacher (5 gen + clustering): ~{t_teacher:.0f}s" if t_teacher else "  Full energy teacher: ~60-120s (5 generations + LLM clustering)")
print(f"  SLT probe (1 generation + 1 forward pass): ~5-15s")
print(f"  TBG probe (0 generations + 1 forward pass): ~0.5-2s  <- fastest")

Efficiency comparison (wall-clock, single example):

   (engine not available in this run — estimates only)

Summary (estimates):
  Full energy teacher: ~60-120s (5 generations + LLM clustering)
  SLT probe (1 generation + 1 forward pass): ~5-15s
  TBG probe (0 generations + 1 forward pass): ~0.5-2s  <- fastest


## Section 16 — Save Trained Probes

In [24]:
probe_bundle = {
    'model_id':                'meta-llama/Llama-3.1-8B-Instruct',
    'dataset':                 'trivia_qa',
    'num_train_records':       D_train.N,
    
    # Binarization thresholds (computed on train split)
    'energy_threshold':        T_energy,
    'entropy_threshold':       T_entropy,
    
    # Best layer ranges
    'best_energy_slt_range':   best_energy_slt_range,
    'best_energy_tbg_range':   best_energy_tbg_range,
    'best_entropy_slt_range':  best_entropy_slt_range,
    'best_entropy_tbg_range':  best_entropy_tbg_range,
    
    # Trained probes and scalers
    'slt_energy_probe':        trained_probes['slt_energy']['probe'],
    'slt_energy_scaler':       trained_probes['slt_energy']['scaler'],
    'tbg_energy_probe':        trained_probes['tbg_energy']['probe'],
    'tbg_energy_scaler':       trained_probes['tbg_energy']['scaler'],
    'slt_entropy_probe':       trained_probes['slt_entropy']['probe'],
    'slt_entropy_scaler':      trained_probes['slt_entropy']['scaler'],
    'tbg_entropy_probe':       trained_probes['tbg_entropy']['probe'],
    'tbg_entropy_scaler':      trained_probes['tbg_entropy']['scaler'],
    
    # Per-layer AUROC results (for inspection)
    'layer_auroc_table': {
        'energy_slt':  energy_slt_aurocs,
        'energy_tbg':  energy_tbg_aurocs,
        'entropy_slt': entropy_slt_aurocs,
        'entropy_tbg': entropy_tbg_aurocs,
    },
    
    # In-distribution test results
    'id_test_results': id_results,
}

output_path = os.path.join(MODELS_DIR, 'probes_llama3-8b_triviaqa.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(probe_bundle, f)

size_kb = os.path.getsize(output_path) / 1024
print(f"Probe bundle saved: {output_path}")
print(f"File size: {size_kb:.0f} KB")
print()
print("Contents:")
for k, v in probe_bundle.items():
    if hasattr(v, '__class__') and 'LogisticRegression' in type(v).__name__:
        print(f"  {k}: LogisticRegression")
    elif isinstance(v, list):
        print(f"  {k}: list[{len(v)}]")
    else:
        print(f"  {k}: {repr(v)[:60]}")

print()
print("Ready for Phase 5: backend/engine.py inference integration")

Probe bundle saved: d:\Github Repositories\SemanticEnergy\backend\models\probes_llama3-8b_triviaqa.pkl
File size: 2052 KB

Contents:
  model_id: 'meta-llama/Llama-3.1-8B-Instruct'
  dataset: 'trivia_qa'
  num_train_records: 400
  energy_threshold: np.float64(0.7503966276628008)
  entropy_threshold: np.float64(0.2051649935096553)
  best_energy_slt_range: (17, 21)
  best_energy_tbg_range: (28, 32)
  best_entropy_slt_range: (20, 24)
  best_entropy_tbg_range: (21, 25)
  slt_energy_probe: LogisticRegression
  slt_energy_scaler: StandardScaler()
  tbg_energy_probe: LogisticRegression
  tbg_energy_scaler: StandardScaler()
  slt_entropy_probe: LogisticRegression
  slt_entropy_scaler: StandardScaler()
  tbg_entropy_probe: LogisticRegression
  tbg_entropy_scaler: StandardScaler()
  layer_auroc_table: {'energy_slt': [0.38048245614035087, 0.4758771929824561, 0.4
  id_test_results: {'slt_energy': {'mean': 0.6666666666666666, 'lo': np.float64

Ready for Phase 5: backend/engine.py inference integrati